In [ ]:
# Instala todas as bibliotecas necessárias para o projeto
!pip install plotly joblib fastapi uvicorn nest-asyncio pyngrok optuna mlflow tensorflow scikit-learn --quiet

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import optuna
import mlflow as mlf

# Confirma versões instaladas
print(f"TensorFlow: {tf.__version__}")
print(f"Keras:      {tf.keras.__version__}")

In [ ]:
DATASET_PATH = "diabetes_dataset_preprocessed.csv"
SCALER_PATH  = "scaler.pkl"
MODEL_PATH   = "diebetes.keras"

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Shape do dataset:", df.shape)
print("\nDistribuição da variável-alvo:")
print(df['diagnosed_diabetes'].value_counts())
df.head()

In [ ]:
X = df.drop(columns=['diagnosed_diabetes', 'diabetes_stage'])
y = df['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, SCALER_PATH)

print(f"Treino: {X_train_sc.shape} | Teste: {X_test_sc.shape}")
print(f"Número de features: {X_train_sc.shape[1]}")

# Rede Neural 


In [ ]:
input_dim = X_train_sc.shape[1]  # número de features

# ── Construção do modelo sequencial ────────────────────────────────────────
model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),   # camada oculta 1
    layers.Dropout(0.2),                   # regularização para evitar overfitting
    layers.Dense(32, activation='relu'),   # camada oculta 2
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # saída binária (probabilidade)
])

# ── Compilação ──────────────────────────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
class_weights = {0: 1.0, 1: 2.0}

history = model.fit(
    X_train_sc, y_train,
    class_weight=class_weights,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

model.save("modelo_diabetesv2.h5")
model.save(MODEL_PATH)
print("\n✅ Modelo salvo!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Treino')
axes[0].plot(history.history['val_loss'], label='Validação')
axes[0].set_title('Loss por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Treino')
axes[1].plot(history.history['val_accuracy'], label='Validação')
axes[1].set_title('Acurácia por Época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].legend()

plt.tight_layout()
plt.show()

## relu + sigmoid + binary_croos


In [ ]:
input_dim = x_train.shape[1]

# Criar e aplicar scaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
# 2. Defina o modelo usando essa variável
model = models.Sequential()

# Agora o input_shape será automaticamente (35,)
model.add(layers.Dense(64, activation='relu', input_shape=(input_dim,)))

model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))
weights = {0: 1.0, 1: 2.0}
# Recompile e tente o fit novamente
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, class_weight=weights, epochs=40, batch_size=32, validation_split=0.2)

# No seu notebook de treino:
model.save("modelo_diabetes.h5")
model.save("diebetes.keras")

joblib.dump(scaler, "scaler2.pkl")

In [ ]:
# ── Predições no conjunto de teste ──────────────────────────────────────────
pred_proba = model.predict(X_test_sc).flatten()  
pred_class = (pred_proba > 0.5).astype(int)      

# ── Erros sobre as probabilidades ───────────────────────────────────────────
mse  = np.mean((y_test.values - pred_proba) ** 2)
mae  = np.mean(np.abs(y_test.values - pred_proba))
rmse = np.sqrt(mse)

# ── Métricas de classificação ────────────────────────────────────────────────
accuracy  = metrics.accuracy_score(y_test, pred_class)
f1        = metrics.f1_score(y_test, pred_class)
precision = metrics.precision_score(y_test, pred_class)
recall    = metrics.recall_score(y_test, pred_class)

print("─" * 40)
print("       MÉTRICAS DE ERRO (Probabilidades)")
print("─" * 40)
print(f"  MSE  (Erro Quadrático Médio):  {mse:.4f}")
print(f"  MAE  (Erro Absoluto Médio):    {mae:.4f}")
print(f"  RMSE (Raiz do MSE):            {rmse:.4f}")
print()
print("─" * 40)
print("       MÉTRICAS DE CLASSIFICAÇÃO")
print("─" * 40)
print(f"  Acurácia:   {accuracy:.4f}")
print(f"  F1-Score:   {f1:.4f}")
print(f"  Precisão:   {precision:.4f}")
print(f"  Recall:     {recall:.4f}")

In [ ]:
# ── Curva ROC ───────────────────────────────────────────────────────────────
fpr, tpr, _ = metrics.roc_curve(y_test, pred_proba)
auc_score   = metrics.auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}', linewidth=2)
plt.plot([0, 1], [0, 1], 'r--', label='Aleatório')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC — Classificação de Diabetes')
plt.legend()
plt.grid(alpha=0.3)

os.makedirs('plots', exist_ok=True)
plt.savefig('plots/ROC_curve.png', dpi=150)
plt.show()
print(f"AUC-ROC: {auc_score:.4f}")

In [ ]:
# ── Matriz de Confusão ──────────────────────────────────────────────────────
cm = metrics.confusion_matrix(y_test, pred_class)
disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sem Diabetes', 'Com Diabetes'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

# ── Carrega modelo e scaler salvos ───────────────────────────────────────────
api_model  = tf.keras.models.load_model(MODEL_PATH)
api_scaler = joblib.load(SCALER_PATH)

# ── Instância da aplicação ───────────────────────────────────────────────────
app = FastAPI(
    title="API de Diagnóstico de Diabetes",
    description="Envia as features do paciente e recebe a probabilidade de diabetes.",
    version="1.0.0"
)

class InputData(BaseModel):
    features: list  # lista com N valores numéricos

@app.get("/")
def root():
    return {"message": "API de Diabetes rodando ✅"}

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True}

@app.post("/predict")
def predict(data: InputData):
    """Recebe lista de features e retorna probabilidade e classe predita."""
    try:
        entrada = np.array(data.features).reshape(1, -1)
        entrada = api_scaler.transform(entrada)      
        prob    = api_model.predict(entrada)
        return {
            "probabilidade": float(prob[0][0]),
            "diabetes":      int(prob[0][0] > 0.5)
        }
    except Exception as e:
        return {"erro": str(e)}